# r-doc-mcp — agent with tool calling

Starts the API, then runs a [deepagents](https://github.com/langchain-ai/deepagents) agent
(same pattern as `langgraph_test/deep_agents.ipynb`): an LLM built from the `COMPLETION_*` env vars
gets `search_docs` as a LangChain tool and decides when to call it.

Run top to bottom. The last cell shuts the API down.
Needs a pushed `.chroma_db` **and** a reachable completion provider (see `.env`) — the service is search-only, so the agent brings its own LLM.
The MCP server itself is covered separately by `tests/test_mcp.py`.

The example questions are about BDC: run with `CONFIG_DIR=examples/bdc` and a database built from r-doc-builder's `examples/bdc/sources.yaml`, or replace them with questions about your own project.

In [ ]:
import socket, subprocess, sys, tempfile, time
from pathlib import Path

import httpx

# a live server keeps its old code and its old .env forever, so re-running this cell must
# replace the previous one rather than leak it and leave you querying a stale process
try:
    api.terminate()
    api.wait(timeout=10)
    print("replaced the previously started API")
except NameError:
    pass

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
with socket.socket() as s:  # let the OS pick a free port so an already-running API doesn't clash
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]
API_URL = f"http://127.0.0.1:{PORT}"

# a file, not a PIPE: nothing drains a pipe here, and uvicorn deadlocks once its buffer fills
LOG = Path(tempfile.gettempdir()) / f"r_api_{PORT}.log"
api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "r_doc_mcp.api:app", "--port", str(PORT)],
    cwd=ROOT, stdout=LOG.open("w"), stderr=subprocess.STDOUT,
)

for _ in range(120):  # first request loads the embedding + chroma clients, so be patient
    if api.poll() is not None:
        raise RuntimeError(f"API died:\n{LOG.read_text()}")
    try:
        httpx.get(f"{API_URL}/health", timeout=2).raise_for_status()
        break
    except Exception:
        time.sleep(1)
else:
    api.terminate()
    raise RuntimeError(f"API never became healthy:\n{LOG.read_text()}")

print("API up at", API_URL, "| server log:", LOG)

## Model + search tool

The tool wraps `POST /search`; the model comes from `.env` (azure / openai / vllm / ollama).

Why a hand-written `@tool` instead of loading the MCP tool via `langchain-mcp-adapters`:
the MCP stdio client needs a real-file `errlog` under ipykernel on Windows (`sys.stderr`
has no `fileno()`), which the adapters don't expose. The MCP server itself is covered by
`tests/test_mcp.py` — the tool below reuses `mcp_server.tool_description()` so the two never drift.

In [ ]:
assert "API_URL" in globals(), "run the boot cell (first code cell) first — it starts the API"

sys.path.insert(0, str(ROOT))  # notebook runs from tests/, the package lives at the repo root

from langchain.tools import tool

import os

from langchain_openai import AzureChatOpenAI, ChatOpenAI

from r_doc_mcp.config import SEARCH_K, doc_types
from r_doc_mcp.mcp_server import tool_description

PROJECT = doc_types()["project"]  # from <CONFIG_DIR>/doc_types.yaml (CONFIG_DIR=examples/bdc for the BDC data)

# the service is search-only — the agent brings its own LLM (COMPLETION_* env vars)
_url, _model = os.getenv("COMPLETION_URL"), os.getenv("COMPLETION_MODEL", "gpt-4o-mini")
if _url and "azure.com" in _url:
    model = AzureChatOpenAI(azure_endpoint=_url, azure_deployment=_model, temperature=0,
                            api_version=os.getenv("AZURE_API_VERSION", "2024-10-21"),
                            api_key=os.getenv("AZURE_OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY"))
else:  # plain OpenAI, or any OpenAI-compatible endpoint (vLLM etc.)
    model = ChatOpenAI(base_url=_url, model=_model, temperature=0,
                       api_key=os.getenv("OPENAI_API_KEY") or "EMPTY")


def _search_docs(query: str, k: int = SEARCH_K, mode: str = "embedding", doc_type: str | None = None,
                 date_from: str | None = None, date_to: str | None = None) -> str:
    resp = httpx.post(f"{API_URL}/search",
                      json={"query": query, "k": k, "mode": mode, "doc_type": doc_type,
                            "date_from": date_from, "date_to": date_to}, timeout=120)
    resp.raise_for_status()
    return "\n\n".join(
        f"[{h['metadata'].get('doc_type')}] {h['metadata'].get('title') or h['metadata'].get('source')}"
        f" ({h['metadata'].get('datetime', 'n.d.')}, score {h['score']:.2f})\n{h['content'][:1200]}"
        for h in resp.json()
    ) or "no results"


_search_docs.__doc__ = tool_description()  # the help text the MCP server publishes, built from doc_types.yaml
search_docs = tool("search_docs")(_search_docs)

In [ ]:
import datetime

from deepagents import create_deep_agent

agent = create_deep_agent(
    model,
    [search_docs],
    system_prompt=(
        # without today's date the model can't turn "recent" into a sensible date_from;
        # the date_from rule lives here because small models ignore it in tool docstrings
        f"Today is {datetime.date.today().isoformat()}. "
        f"You answer questions about {PROJECT}. "
        "Use the search_docs tool to find relevant documentation before answering, "
        "and cite the sources you used. "
        "Search results are ranked by relevance, NOT date. For questions about recent or "
        "latest events/news you MUST pass date_from (start about a year back and widen the "
        "range if nothing returns), then pick the newest date among the results."
    ),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is PIC-SURE and what can I do with it in BDC?"}]}
)

for m in result["messages"]:
    m.pretty_print()

In [ ]:
# the tool schema exposes doc_type — see whether the model narrows the search on its own
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Find a recent BDC community hours event. When was it held?"}]}
)

for m in result["messages"]:
    m.pretty_print()

In [ ]:
try:
    api.terminate()
    api.wait(timeout=10)
    print("API stopped. Server log:\n")
    print(LOG.read_text())
except NameError:
    print("no API to stop — the boot cell hasn't run in this kernel")